# Toy representations

Choose a completed run below. The first plot shows training/evaluation losses;
PCA visualizes the saved evaluation embeddings. If a frozen-encoder decoder has
been trained for this run, the last cell compares observations and reconstructions.


In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "toy/train.py").is_file())
TOY = ROOT / "toy"
sys.path.insert(0, str(TOY))

import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml
from sklearn.decomposition import PCA

RUN = "dot_smoke"  # dot, structured, sprite, or their _smoke runs
DIRECTORY = TOY / "outputs" / RUN
AVAILABLE = (DIRECTORY / "model.pt").is_file()
if not AVAILABLE:
    print(f"Train a model first; no model.pt in {DIRECTORY}")


In [ ]:
if AVAILABLE:
    history = torch.load(DIRECTORY / "train_history.pt", weights_only=False)
    exported = torch.load(DIRECTORY / "embeddings.pt", weights_only=False)
    final_epoch = max(exported["snapshots"])
    z = exported["snapshots"][final_epoch]["z_t"].numpy()
    positions = exported["positions"].numpy().reshape(len(z), -1)
    pca = PCA(n_components=3).fit(z)
    projected = pca.transform(z)
    fig = plt.figure(figsize=(8, 3), layout="constrained")
    ax = fig.add_subplot(1, 2, 1)
    for split in ["train", "eval"]:
        values = history[split]
        ax.plot(values["epoch"], values["total"], label=split)
    ax.set(xlabel="Epoch", ylabel="Forward + inverse loss")
    ax.legend(frameon=False)
    geometry = fig.add_subplot(1, 2, 2, projection="3d")
    points = geometry.scatter(*projected.T, c=positions[:, 0], cmap="viridis", s=4, linewidths=0)
    geometry.set(xlabel="PC1", ylabel="PC2", zlabel="PC3")
    fig.colorbar(points, ax=geometry, shrink=0.6, label="First position coordinate")
    plt.show()
    print("Explained variance:", pca.explained_variance_ratio_)


In [ ]:
if AVAILABLE and (DIRECTORY / "decoder.pt").is_file():
    from train import build_world, evaluation_seed
    from models import CNNEncoder, CNNDecoder

    cfg = yaml.safe_load((DIRECTORY / "config.yaml").read_text())
    world, dataset_type, _ = build_world(cfg["dataset"])
    dataset = dataset_type(world, num_samples=8,
                           seed=evaluation_seed(cfg["dataset"], cfg["training"]["seed"]))
    observations = torch.stack([dataset[i][0] for i in range(8)])
    encoder = CNNEncoder(latent_dim=cfg["model"]["latent_dim"], image_size=world.image_size).eval()
    decoder = CNNDecoder(latent_dim=cfg["model"]["latent_dim"], image_size=world.image_size).eval()
    encoder.load_state_dict(torch.load(DIRECTORY / "model.pt", weights_only=True)["encoder"])
    decoder.load_state_dict(torch.load(DIRECTORY / "decoder.pt", weights_only=True))
    with torch.no_grad():
        reconstructed = decoder(encoder(observations))
    fig, axes = plt.subplots(2, 8, figsize=(8, 2), layout="constrained")
    for i in range(8):
        axes[0, i].imshow(observations[i].permute(1, 2, 0).numpy(), cmap="gray", vmin=0, vmax=1)
        axes[1, i].imshow(reconstructed[i].permute(1, 2, 0).numpy(), cmap="gray", vmin=0, vmax=1)
        axes[0, i].set_axis_off()
        axes[1, i].set_axis_off()
    plt.show()
